# Лаборатория: Spark + MinIO + ClickHouse

Этот ноутбук проверяет всю цепочку:

1. Spark в локальном режиме (`local[*]`) внутри этого контейнера.
2. Запись/чтение Parquet в MinIO по S3A (`s3a://warehouse/...`).
3. Создание таблиц в кластере ClickHouse `lab_cluster` (2 шарда × 2 реплики).
4. Запись DataFrame в ClickHouse через JDBC.

Все креды: `admin` / `admin` (MinIO — `admin` / `adminadmin`).


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("de-lab")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "4")
    # --- S3A / MinIO ---
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "adminadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

spark.version


'3.5.3'

In [ ]:
from pyspark.sql import functions as F

df = (
    spark.range(0, 1000)
    .withColumn("name", F.concat(F.lit("row-"), F.col("id")))
    .withColumn("value", F.round(F.rand(seed=42) * 100, 3))
    .withColumn("source", F.lit("spark"))
)

target = "s3a://warehouse/demo/events"
df.write.mode("overwrite").parquet(target)
print("written:", target)

back = spark.read.parquet(target)
print("rows in MinIO:", back.count())
back.show(5, truncate=False)


## ClickHouse: DDL на весь кластер

`ON CLUSTER lab_cluster` выполняет DDL на всех 4 нодах через ClickHouse Keeper.
Таблица `lab.events` — `ReplicatedMergeTree` (по реплике на шард), `lab.events_all` — `Distributed` поверх неё.


In [ ]:
import clickhouse_connect

ch = clickhouse_connect.get_client(host="ch1-1", port=8123, username="admin", password="admin")

ddl = [
    "CREATE DATABASE IF NOT EXISTS lab ON CLUSTER lab_cluster",
    '''
    CREATE TABLE IF NOT EXISTS lab.events ON CLUSTER lab_cluster
    (
        id     Int64,
        name   String,
        value  Float64,
        source String
    )
    ENGINE = ReplicatedMergeTree('/clickhouse/tables/{shard}/lab/events', '{replica}')
    ORDER BY id
    ''',
    '''
    CREATE TABLE IF NOT EXISTS lab.events_all ON CLUSTER lab_cluster
    AS lab.events
    ENGINE = Distributed('lab_cluster', 'lab', 'events', rand())
    ''',
]
for stmt in ddl:
    ch.command(stmt, settings={"distributed_ddl_task_timeout": 120})

print(ch.query("SELECT cluster, shard_num, replica_num, host_name FROM system.clusters WHERE cluster = 'lab_cluster'").result_rows)


## Spark -> ClickHouse по JDBC

Пишем в шард 1 (нода `ch1-1`). Реплика `ch1-2` подхватит данные автоматически,
а `lab.events_all` на любой ноде покажет общую картину.


In [ ]:
jdbc_url = "jdbc:clickhouse://ch1-1:8123/lab"
jdbc_props = {
    "user": "admin",
    "password": "admin",
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
}

(df.select("id", "name", "value", "source")
   .write
   .mode("append")
   .option("driver", jdbc_props["driver"])
   .option("user", jdbc_props["user"])
   .option("password", jdbc_props["password"])
   .jdbc(jdbc_url, "events"))

print("written to ClickHouse")


## Проверка репликации и распределённого чтения

In [ ]:
ch1_2 = clickhouse_connect.get_client(host="ch1-2", port=8123, username="admin", password="admin")
ch2_1 = clickhouse_connect.get_client(host="ch2-1", port=8123, username="admin", password="admin")

print("shard1 replica ch1-1:", ch.query("SELECT count() FROM lab.events").first_row[0])
print("shard1 replica ch1-2:", ch1_2.query("SELECT count() FROM lab.events").first_row[0])
print("distributed view (via ch2-1):", ch2_1.query("SELECT count() FROM lab.events_all").first_row[0])


In [ ]:
jdbc_read = (
    spark.read.format("jdbc")
    .option("url", "jdbc:clickhouse://ch2-1:8123/lab")
    .option("dbtable", "events_all")
    .option("user", "admin")
    .option("password", "admin")
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
    .load()
)
jdbc_read.show(5, truncate=False)
print("rows read back via Spark:", jdbc_read.count())


## Полезные ссылки

- MinIO Console: http://192.168.0.13:9001 (admin / adminadmin)
- Airflow: http://192.168.0.13:8080 (admin / admin)
- Spark UI: http://192.168.0.13:4040 (пока жива эта сессия)
- ClickHouse HTTP: http://192.168.0.13:8123/play (admin / admin)

Полный справочник адресов — в `CONNECTIONS.md`.
